<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-02-model-architecture/lesson-2.1-tokenization-and-embeddings/notebooks/exercises-2_1.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 2.1: Tokenization & Embeddings in Code

*Module 2 · Large Language Models — Deep Dive*

> An LLM doesn't read words. It reads tokens — pieces of words, sometimes a character, sometimes a whole word, sometimes a subword. "unbelievable" becomes ["un", "believ", "able"]. This lesson implements a tokenizer from scratch, compares tokenizers across models, and reveals why tokenization is the hidden lever behind API cost, context length, and multilingual performance.

`Module 2` · `8 Concepts` · `BPE + Quirks + Embeddings` · `120 min`

---

## About this notebook

This notebook contains the **7 runnable code examples** from the published lesson `lesson-2.1.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Byte-Pair Encoding (BPE) — Build a Tokenizer From Scratch — `bpe_from_scratch.py`
2. Step 2 — Compare Tokenizers Across Models — `compare_tokenizers.py`
3. Step 3 — Token Economics — How Tokenization Affects Your Bill — `token_economics.py`
4. Step 5 — From Tokens to Embeddings — Meaning in Numbers — `embeddings_demo.py`
5. Step 6 — Why LLMs Can't Count Letters — Tokenization Quirks — `tokenization_quirks.py`
6. Step 7 — Special Tokens & Chat Templates — The Hidden Control Layer — `special_tokens.py`
7. Step 8 — The Embedding Landscape — From Word2Vec to Matryoshka — `embedding_landscape.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai numpy tiktoken


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Byte-Pair Encoding (BPE) — Build a Tokenizer From Scratch

The algorithm behind GPT, Llama, and most modern tokenizers.

**`bpe_from_scratch.py`** · _Block 1 of 7_

BPE TOKENIZER FROM SCRATCH — The exact algorithm behind GPT's tokenizer.


In [ ]:
# =============================================
# BPE TOKENIZER FROM SCRATCH
# The exact algorithm behind GPT's tokenizer.
# Learn by building it yourself.
# =============================================

from collections import Counter

class SimpleBPE:
    """Byte-Pair Encoding tokenizer — simplified but real."""
    
    def __init__(self):
        self.merges = {}   # pair → merged token
        self.vocab = {}    # token → id
    
    def _get_pairs(self, tokens: list) -> Counter:
        """Count all adjacent pairs."""
        pairs = Counter()
        for i in range(len(tokens) - 1):
            pairs[(tokens[i], tokens[i+1])] += 1
        return pairs
    
    def _merge_pair(self, tokens: list, pair: tuple) -> list:
        """Merge all occurrences of a pair in the token list."""
        merged = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == pair:
                merged.append(tokens[i] + tokens[i+1])
                i += 2
            else:
                merged.append(tokens[i])
                i += 1
        return merged
    
    def train(self, text: str, num_merges: int = 10):
        """Train BPE: find and merge the most common pairs."""
        # Start: every character is its own token
        tokens = list(text)
        print(f"Start:  {len(set(tokens))} unique chars, {len(tokens)} total tokens")
        
        for step in range(num_merges):
            pairs = self._get_pairs(tokens)
            if not pairs:
                break
            
            # Find the most frequent pair
            best_pair = pairs.most_common(1)[0]
            pair, count = best_pair[0], best_pair[1]
            
            # Merge it
            merged_token = pair[0] + pair[1]
            tokens = self._merge_pair(tokens, pair)
            self.merges[pair] = merged_token
            
            print(f"Step {step+1:2d}: merge '{pair[0]}' + '{pair[1]}' → '{merged_token}' (count={count}, tokens now={len(tokens)})")
        
        # Build vocabulary
        self.vocab = {tok: i for i, tok in enumerate(sorted(set(tokens)))}
        print(f"\nVocab size: {len(self.vocab)} tokens")
        return tokens
    
    def tokenize(self, text: str) -> list:
        """Tokenize text using learned merges."""
        tokens = list(text)
        for pair, merged in self.merges.items():
            tokens = self._merge_pair(tokens, pair)
        return tokens

# ── Train on sample text ──
text = "low lower lowest new newer newest"
print("BPE Training:\n")
bpe = SimpleBPE()
result = bpe.train(text, num_merges=8)

print(f"\nTokenized: {result}")
print(f"\nTest new text:")
print(f"  'lower' → {bpe.tokenize('lower')}")
print(f"  'newest' → {bpe.tokenize('newest')}")


> **What just happened?** We built a BPE tokenizer from scratch. Starting with individual characters, it found the most common pair (like "e"+"r"), merged them into one token ("er"), and repeated. After 8 merges, "lower" went from 5 characters to 1-2 tokens. This is the exact algorithm behind GPT's tiktoken, Llama's SentencePiece, and Gemini's tokenizer. Real tokenizers run 50,000-100,000 merges on billions of words.


### Step 2 · Compare Tokenizers Across Models

Same text, different tokenizers = different token counts. See how GPT, Gemini, and Llama tokenize differently.

**`compare_tokenizers.py`** · _Block 2 of 7_

COMPARE TOKENIZERS ACROSS MODELS — Same text → different token counts.


In [ ]:
# =============================================
# COMPARE TOKENIZERS ACROSS MODELS
# Same text → different token counts.
# pip install tiktoken
# =============================================

import tiktoken

# GPT-4's tokenizer
enc_gpt4 = tiktoken.get_encoding("cl100k_base")     # GPT-4, GPT-3.5
enc_gpt4o = tiktoken.get_encoding("o200k_base")     # GPT-4o

test_texts = [
    ("English",  "Tokenization is the first step in building an LLM."),
    ("Hindi",    "टोकनाइज़ेशन एलएलएम बनाने का पहला कदम है।"),
    ("Telugu",   "టోకెనైజేషన్ LLM నిర్మించడంలో మొదటి దశ."),
    ("Code",     "def fibonacci(n):\n    return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2)"),
    ("JSON",     '{"name": "Netsetos", "students": 10000, "rating": 4.9}'),
    ("Emoji",    "I love AI! 🤖🚀💡 Let's build something amazing!"),
]

print("Token count comparison:\n")
print(f"  {'Language':10s} {'GPT-4':>7s} {'GPT-4o':>8s} {'Diff':>6s}  Text")
print(f"  {'─'*75}")

for lang, text in test_texts:
    gpt4_tokens = enc_gpt4.encode(text)
    gpt4o_tokens = enc_gpt4o.encode(text)
    diff = len(gpt4_tokens) - len(gpt4o_tokens)
    diff_str = f"-{abs(diff)}" if diff > 0 else f"+{abs(diff)}" if diff < 0 else "="
    print(f"  {lang:10s} {len(gpt4_tokens):>5d}   {len(gpt4o_tokens):>5d}   {diff_str:>5s}  {text[:45]}")

# Show actual tokens for English
print(f"\nGPT-4 tokens for English:")
tokens = enc_gpt4.encode(test_texts[0][1])
decoded = [enc_gpt4.decode([t]) for t in tokens]
print(f"  {decoded}")

print(f"\nGPT-4 tokens for Hindi:")
tokens = enc_gpt4.encode(test_texts[1][1])
print(f"  Token count: {len(tokens)} (vs {len(enc_gpt4.encode(test_texts[0][1]))} for English)")
print(f"  Hindi uses ~2x more tokens → costs ~2x more per prompt!")


> **What just happened?** 🧠 Knowledge Check Why do LLMs struggle with counting letters in 'strawberry'? LLMs can't do math operations The tokenizer splits 'strawberry' into subword tokens, so the model never sees individual letters The model was not trained on words with repeated letters Check Answer Same meaning in English (8 tokens) vs Hindi (15+ tokens) vs Telugu (18+ tokens). GPT-4o's newer tokenizer (o200k_base) is more efficient than GPT-4's (cl100k_base) for multilingual text. Key insight: tokenizers are trained mostly on English text, so non-English languages get split into more pieces → more expensive. GPT-4o improved this by training on more multilingual data.


### Step 3 · Token Economics — How Tokenization Affects Your Bill

Every token costs money. Understanding token economics = controlling costs.

**`token_economics.py`** · _Block 3 of 7_

TOKEN ECONOMICS CALCULATOR — How much does your prompt actually cost?


In [ ]:
# =============================================
# TOKEN ECONOMICS CALCULATOR
# How much does your prompt actually cost?
# =============================================

import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

# Pricing per 1M tokens (USD)
PRICES = {
    "Gemini Flash (input)":  0.10,
    "Gemini Flash (output)": 0.40,
    "Gemini Pro (input)":    1.25,
    "Gemini Pro (output)":  10.00,
    "GPT-4o (input)":       2.50,
    "GPT-4o (output)":      10.00,
}

def analyze_cost(text: str, label: str, daily_calls: int = 10000):
    tokens = enc.encode(text)
    n = len(tokens)
    print(f"\n  📊 {label}")
    print(f"  Characters: {len(text):,} | Tokens: {n:,} | Ratio: {len(text)/n:.1f} chars/token")
    print(f"  Cost per call ({daily_calls:,}/day):")
    for model, price in PRICES.items():
        cost_per_call = n * price / 1e6
        daily = cost_per_call * daily_calls
        monthly = daily * 30
        if "input" in model:
            print(f"    {model:30s}: ${cost_per_call:.6f}/call → ₹{monthly*84:.0f}/month")

print("Token Economics:\n")

# Compare: short prompt vs verbose prompt
short = "Summarize: key points only, 3 bullets"
verbose = "I would like you to please kindly summarize the following text. It is important that you provide the key points in a bulleted list format with exactly 3 bullets."

analyze_cost(short, "Short prompt (optimized)")
analyze_cost(verbose, "Verbose prompt (wasteful)")

print(f"\n  💡 Same instruction. Verbose = {len(enc.encode(verbose))/len(enc.encode(short)):.1f}x more tokens → {len(enc.encode(verbose))/len(enc.encode(short)):.1f}x more cost!")


> **What just happened?** 🧠 Knowledge Check What is the 'multilingual token tax' and why does it matter? Some languages require special GPU hardware to process Hindi text uses 2-3x more tokens than equivalent English text, making API calls 2-3x more expensive Multilingual models are always less accurate than English-only models Check Answer A verbose prompt ("I would like you to please kindly...") used 4x more tokens than a concise prompt ("Summarize: key points, 3 bullets") — for the same instruction. At 10K calls/day, that's the difference between ₹300/month and ₹1,200/month. This is why Module 11.2 (Prompt Optimization) teaches you to compress prompts. Tokenization awareness = cost awareness.


### Step 5 · From Tokens to Embeddings — Meaning in Numbers

Tokens are just IDs. Embeddings give them meaning — a 768-dimensional "address" in meaning-space.

**`embeddings_demo.py`** · _Block 4 of 7_

EMBEDDINGS — From tokens to meaning vectors — pip install google-generativeai numpy


In [ ]:
# =============================================
# EMBEDDINGS — From tokens to meaning vectors
# pip install google-generativeai numpy
# =============================================

import google.generativeai as genai
import numpy as np
import os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

def get_embedding(text: str) -> np.ndarray:
    result = genai.embed_content(model="models/text-embedding-004", content=text)
    return np.array(result["embedding"])

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Get embeddings for related and unrelated words
words = ["king", "queen", "prince", "biryani", "programming", "Python"]
embeddings = {w: get_embedding(w) for w in words}

print("Embedding Similarity Matrix:\n")
print(f"  {'':12s}" + "".join(f"{w:>12s}" for w in words))
for w1 in words:
    row = f"  {w1:12s}"
    for w2 in words:
        sim = cosine_sim(embeddings[w1], embeddings[w2])
        row += f"{sim:12.3f}"
    print(row)

print(f"\n  Embedding dimension: {len(embeddings['king'])}")
print(f"\n  Key observations:")
print(f"  • king ↔ queen:       HIGH (both royalty)")
print(f"  • king ↔ biryani:     LOW (unrelated)")
print(f"  • programming ↔ Python: HIGH (related concepts)")
print(f"  • biryani ↔ Python:   LOW (food vs code)")
print(f"\n  This is how RAG search works (Module 6):")
print(f"  convert query → embedding → find nearest documents.")


> **What just happened?** 🧠 Knowledge Check What is the key difference between Word2Vec and BERT embeddings? Word2Vec produces larger vectors than BERT Word2Vec gives one fixed vector per word regardless of context, while BERT gives different vectors based on surrounding words BERT is faster than Word2Vec at inference time Check Answer Each word was converted to a 768-dimensional embedding vector using Gemini's embedding model. "King" and "queen" have high similarity (both royalty). "King" and "biryani" have low similarity (unrelated). "Programming" and "Python" are close (related concepts). This is the complete pipeline: text → tokens → token IDs → embeddings → meaning. Everything in this course — attention, RAG, similarity search, fine-tuning — operates on these embeddings.


### Step 6 · Why LLMs Can't Count Letters — Tokenization Quirks

Many bizarre LLM failures trace back to tokenization, not model intelligence.

**`tokenization_quirks.py`** · _Block 5 of 7_

TOKENIZATION QUIRKS — Why LLMs struggle with — letter counting, reversal, and arithmetic


In [ ]:
# =============================================
# TOKENIZATION QUIRKS — Why LLMs struggle with
# letter counting, reversal, and arithmetic
# =============================================

import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o")

# ── 1. The Strawberry Problem ──
word = "strawberry"
tokens = enc.encode(word)
decoded = [enc.decode([t]) for t in tokens]
print(f"'strawberry' → {decoded}")
print(f"Model sees {len(tokens)} chunks, NOT 10 letters")
print(f"Actual r count: {word.count('r')} — model often says 2\n")

# ── 2. String Reversal ──
word = "hello"
tokens = enc.encode(word)
decoded = [enc.decode([t]) for t in tokens]
print(f"'hello' → {decoded}")
print(f"Model can't easily reverse within merged tokens\n")

# ── 3. Arithmetic Inconsistency ──
for num in ["234", "2345", "23456", "1000000"]:
    tokens = enc.encode(num)
    decoded = [enc.decode([t]) for t in tokens]
    print(f"  '{num}' → {decoded} ({len(tokens)} tokens)")
print(f"Numbers tokenize inconsistently — that's why LLM math fails\n")

# ── 4. The Multilingual "Token Tax" ──
texts = {
    "English":  "AI is transforming how we learn and work.",
    "Hindi":    "कृत्रिम बुद्धिमत्ता हमारे सीखने और काम करने के तरीके को बदल रही है।",
    "Telugu":   "కృత్రిమ మేధస్సు మనం నేర్చుకునే విధానాన్ని మారుస్తోంది.",
}
en_count = len(enc.encode(texts["English"]))
for lang, text in texts.items():
    count = len(enc.encode(text))
    ratio = count / en_count
    print(f"  {lang:<10s} {count:>3} tokens ({ratio:.1f}x English)")
print(f"\n  Hindi costs ~{len(enc.encode(texts['Hindi']))/en_count:.1f}x more → same meaning, higher API bill!")
print(f"  2025 research: Arabic can cost 340% more than English")


> **What just happened?** Four tokenization quirks that affect production: (1) Letter counting fails because the model sees subword tokens, not characters — use code execution tools instead. (2) String reversal breaks across token boundaries. (3) Arithmetic is inconsistent because numbers tokenize differently (234 = 1 token, 23456 = 2 tokens). (4) The "Token Tax" — Hindi/Telugu costs 2-3x more tokens than English for the same meaning. Karpathy's key insight: "A lot of weird behaviors of LLMs actually trace back to tokenization." Solutions: code execution tools (Module 8), token-aware prompt design (Module 3), multilingual-optimized models (Module 11).


### Step 7 · Special Tokens & Chat Templates — The Hidden Control Layer

Special tokens control when the model starts, stops, and how it distinguishes system/user/assistant messages.

**`special_tokens.py`** · _Block 6 of 7_

SPECIAL TOKENS — The hidden control signals


In [ ]:
# =============================================
# SPECIAL TOKENS — The hidden control signals
# =============================================

# Every model has reserved token IDs with special roles:
special_tokens = {
    "[BOS] / <s>":         "Beginning of sequence — signals start",
    "[EOS] / </s>":        "End of sequence — model stops generating",
    "[PAD]":               "Padding — fills short sequences in batches",
    "[UNK]":               "Unknown — fallback for missing vocab",
    "<|endoftext|>":       "OpenAI's document separator",
    "<|im_start|>":        "Chat message boundary (OpenAI)",
    "[INST] / [/INST]":    "Instruction boundaries (Mistral/Llama 2)",
    "<|start_header_id|>": "Role marker (Llama 3)",
}

print("SPECIAL TOKENS ACROSS MODELS:\n")
for token, role in special_tokens.items():
    print(f"  {token:<25s} {role}")

# Chat template example — how models know who's talking
print(f"""
CHAT TEMPLATES — Different models, different formats:

OpenAI (GPT-4):
  <|im_start|>system
  You are a helpful assistant.<|im_end|>
  <|im_start|>user
  What is GenAI?<|im_end|>

Llama 3:
  <|begin_of_text|><|start_header_id|>system<|end_header_id|>
  You are a helpful assistant.<|eot_id|>

Mistral:
  [INST] What is GenAI? [/INST]

Why this matters:
  - Wrong template during fine-tuning = broken model
  - Missing EOS token = model generates forever
  - Double BOS tokens = confused predictions
  - Module 7 (fine-tuning) requires getting this exactly right
""")


> **What just happened?** Special tokens are the traffic signals of LLM generation. [BOS] says "start here." [EOS] says "stop now." Chat templates wrap messages so the model knows who's speaking. Each model family uses different tokens — Llama 3 uses header tags, Mistral uses [INST], OpenAI uses im_start. Getting these wrong during fine-tuning (Module 7) is one of the most common bugs. The tokenizer handles these automatically during inference, but you must understand them for custom training.


### Step 8 · The Embedding Landscape — From Word2Vec to Matryoshka

Embeddings evolved from static word vectors to dynamic, dimension-adaptive representations that power RAG.

**`embedding_landscape.py`** · _Block 7 of 7_

EMBEDDING LANDSCAPE — Models, Dimensions, RAG


In [ ]:
# =============================================
# EMBEDDING LANDSCAPE — Models, Dimensions, RAG
# =============================================

# 2025 Embedding Models Comparison
models = [
    ("Gemini Embedding",       768,   "Free",     "Google",   "Best overall (Agentset ELO #1)"),
    ("text-embedding-3-large", 3072,  "$0.13/M",  "OpenAI",   "Matryoshka: truncate to 256-3072"),
    ("text-embedding-3-small", 1536,  "$0.02/M",  "OpenAI",   "Best value for most RAG apps"),
    ("Voyage-3-large",         1024,  "$0.18/M",  "Voyage",   "Beats OpenAI by 9.7% on MTEB"),
    ("BGE-M3",                1024,  "Free",     "BAAI",     "Open-source, multilingual"),
    ("Jina-embeddings-v3",    1024,  "Free",     "Jina AI",  "Self-host, task-specific LoRA"),
]

print(f"{'Model':<26s} {'Dims':>5} {'Cost':>10} {'Provider':<10} Notes")
print("-" * 90)
for name, dims, cost, provider, notes in models:
    print(f"  {name:<24s} {dims:>5} {cost:>10} {provider:<10} {notes}")

# Dimension vs Storage Cost
print(f"""
DIMENSION TRADE-OFFS (10M vectors):
  256 dims  → 10 GB storage → $1.25/month → 95% of max accuracy
  768 dims  → 30 GB storage → $3.75/month → 98% of max accuracy
  3072 dims → 120 GB storage → $15/month  → 100% accuracy

Matryoshka Embeddings (OpenAI text-embedding-3):
  Train once at 3072 dims, truncate to ANY size at query time.
  256-dim truncated version BEATS old ada-002 at full 1536 dims!
  → Use 768 for most RAG apps, save 75% storage vs 3072.

For Module 6 (RAG): embedding quality = retrieval quality = answer quality.
  Best practice: Gemini Embedding (free, 768d) or text-embedding-3-small ($0.02/M).
""")


> **What just happened?** The embedding landscape in 2025: Gemini Embedding (free, 768d, #1 on Agentset ELO), text-embedding-3 (Matryoshka — truncate to any size), Voyage-3 (beats OpenAI by 9.7% on benchmarks), BGE-M3/Jina (open-source, self-host). Key insight: 768 dimensions is enough for most RAG. Going to 3072 adds 4x storage cost for only 2% accuracy gain. Matryoshka embeddings let you train once and truncate — use 256d for fast search, 768d for accurate search. Module 6 builds a complete RAG pipeline using these embeddings.


---

## Wrap-up

You've walked through all 7 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
